# HuBMAP — Cascade Mask R-CNN swap inference

Replaces the original 2023-1st-place's weakest stream (`m0i/m1i`, Mask R-CNN + Swin-S, WBF weight=1) with a self-trained **Cascade Mask R-CNN + Swin-L** (`cascade0/cascade1`, WBF weight=2).

**Stream mix** (10 mmdet streams, no YOLO — YOLO already shown to drag down WBF):

| Stream | Source | Weight |
|---|---|---|
| r0i, r1i | 1st place (RTMDet + CSPNeXt-X) | 2, 2 |
| s0i, s1i | 1st place (RTMDet + Swin-B 3-stage) | 2, 2 |
| sb0i, sb1i | 1st place (RTMDet + Swin-B 5-stage) | 2, 2 |
| y0i, y1i | 1st place (RTMDet variant) | 1, 1 |
| **cascade0, cascade1** | **NEW, self-trained Cascade MaskRCNN + Swin-L fold 0/1** | **2, 2** |

**Required Kaggle datasets** (attach to notebook):
- `hubmap-2023-configs-patched` (legacy mmdet configs `m0i.py`, `r0i.py`, `s0i.py`, `sb0i.py`, `y0i.py`)
- `hubmap-2023-checkpoints` (legacy ckpts: `r0i.pth, r1i.pth, s0i.pth, s1i.pth, sb0i.pth, sb1i.pth, y0i.pth, y1i.pth`)
- `hubmap-2023-modules` (the `hubmap_modules` package with `RandomRotateScaleCrop`, `FastCocoMetric`, etc.)
- `mmdetv3-env`, `vasculature-packages`, `pycocotools-206` (offline wheels — same as legacy notebook)
- `hubmap-cascade-ckpts` (**NEW** — contains `m0ci.py`, `m0c_fold0.pth`, `m0c_fold1.pth` produced by `tools/dump_ckpt.py`)
- Competition data: `hubmap-hacking-the-human-vasculature`

In [ ]:
# Auto-alias Kaggle datasets to flat paths under /kaggle/working/inputs/.
import os, glob, shutil

SYMLINK_DIR = '/kaggle/working/inputs'
os.makedirs(SYMLINK_DIR, exist_ok=True)

SLUG_TO_REAL = {}
for owner_dir in sorted(glob.glob('/kaggle/input/datasets/*')):
    if os.path.isdir(owner_dir):
        for ds in sorted(glob.glob(os.path.join(owner_dir, '*'))):
            if os.path.isdir(ds):
                SLUG_TO_REAL[os.path.basename(ds)] = ds
for ds in sorted(glob.glob('/kaggle/input/competitions/*')):
    if os.path.isdir(ds):
        SLUG_TO_REAL[os.path.basename(ds)] = ds
for ds in sorted(glob.glob('/kaggle/input/*')):
    name = os.path.basename(ds)
    if name in ('datasets', 'competitions'):
        continue
    if os.path.isdir(ds) and name not in SLUG_TO_REAL:
        SLUG_TO_REAL[name] = ds

for slug, real in SLUG_TO_REAL.items():
    flat = os.path.join(SYMLINK_DIR, slug)
    if os.path.lexists(flat):
        try:
            if os.path.islink(flat):
                os.unlink(flat)
            elif os.path.isdir(flat):
                shutil.rmtree(flat)
            else:
                os.unlink(flat)
        except OSError as e:
            print(f'WARN: could not clean existing {flat}: {e}')
    os.symlink(real, flat)

print(f'Created {len(SLUG_TO_REAL)} symlinks under {SYMLINK_DIR}:')
for slug in sorted(SLUG_TO_REAL):
    print(f'  {SYMLINK_DIR}/{slug} -> {SLUG_TO_REAL[slug]}')

In [ ]:
!python --version

!pip install -q --no-index /kaggle/working/inputs/mmdetv3-env/archive/addict-2.4.0-py3-none-any.whl
!pip install -q --no-index /kaggle/working/inputs/vasculature-packages/mmengine-0.8.3-py3-none-any.whl
!pip install -q --no-index /kaggle/working/inputs/mmdetv3-env/archive/mmcv-2.0.0-cp310-cp310-linux_x86_64.whl
!pip install -q --no-index /kaggle/working/inputs/mmdetv3-env/archive/terminaltables-3.1.10-py2.py3-none-any.whl
!pip install -q --no-index /kaggle/working/inputs/pycocotools-206/wheels/pycocotools-2.0.6-cp310-cp310-linux_x86_64.whl
!pip install -q --no-index /kaggle/working/inputs/vasculature-packages/mmdet-3.1.0-py3-none-any.whl
!pip install -q --no-index /kaggle/working/inputs/vasculature-packages/ensemble_boxes-1.0.9-py3-none-any.whl
!pip install -q --no-index /kaggle/working/inputs/vasculature-packages/ordered_set-4.1.0-py3-none-any.whl
!pip install -q --no-index /kaggle/working/inputs/vasculature-packages/model_index-0.1.11-py3-none-any.whl
!pip install -q --no-index /kaggle/working/inputs/vasculature-packages/einops-0.6.1-py3-none-any.whl
!pip install -q --no-index /kaggle/working/inputs/vasculature-packages/mat4py-0.5.0-py2.py3-none-any.whl
!pip install --no-deps --no-index /kaggle/working/inputs/vasculature-packages/mmpretrain-1.0.1-py2.py3-none-any.whl

In [ ]:
# Make hubmap_modules importable (the inference-side custom modules package)
!cp -r /kaggle/working/inputs/hubmap-2023-modules /kaggle/working/hubmap_modules

In [ ]:
# Build a COCO-style metadata JSON for the test images so the mmdet test_dataloader can iterate them.
import glob
import os
import mmengine

def prepare_dataset():
    coco = {
        'info': {},
        'categories': [
            {'id': 0, 'name': 'blood_vessel'},
            {'id': 1, 'name': 'glomerulus'},
            {'id': 2, 'name': 'unsure'},
        ],
        'annotations': [],
    }
    test_imgs = glob.glob('/kaggle/working/inputs/hubmap-hacking-the-human-vasculature/test/*.tif')
    img_infos = []
    for img_id, path in enumerate(test_imgs):
        img_infos.append(dict(id=img_id, width=512, height=512, file_name=os.path.basename(path)))
    coco['images'] = img_infos
    return coco

mmengine.dump(prepare_dataset(), '/kaggle/working/test.json')

In [ ]:
%%writefile tta_test.py
"""Custom horizontal-flip TTA inference for mmdet instance segmentation models
(mmdet's native DetTTAModel doesn't support seg masks).

Runs orig + h-flipped inference, flips boxes back, dedupes by per-class NMS."""
import argparse
import glob
import os

import mmcv
import mmengine
import numpy as np
import torch
from torchvision.ops import nms
from mmdet.apis import init_detector, inference_detector
from mmdet.utils import register_all_modules

parser = argparse.ArgumentParser()
parser.add_argument('cfg')
parser.add_argument('ckpt')
parser.add_argument('--out', required=True)
parser.add_argument('--nms-iou', type=float, default=0.5)
args = parser.parse_args()

register_all_modules()
model = init_detector(args.cfg, args.ckpt, device='cuda')

TEST_IMGS = sorted(glob.glob(
    '/kaggle/working/inputs/hubmap-hacking-the-human-vasculature/test/*.tif'))

test_meta = mmengine.load('/kaggle/working/test.json')
id_by_name = {os.path.basename(im['file_name']): im['id']
              for im in test_meta['images']}

results = []
for path in TEST_IMGS:
    img = mmcv.imread(path)
    H, W = img.shape[:2]
    with torch.no_grad():
        r_orig = inference_detector(model, img)
        img_flip = np.ascontiguousarray(img[:, ::-1, :])
        r_flip = inference_detector(model, img_flip)
    inst_o = r_orig.pred_instances
    inst_f = r_flip.pred_instances
    bb_f = inst_f.bboxes.clone()
    bb_f[:, [0, 2]] = W - bb_f[:, [2, 0]]
    bboxes = torch.cat([inst_o.bboxes, bb_f])
    scores = torch.cat([inst_o.scores, inst_f.scores])
    labels = torch.cat([inst_o.labels, inst_f.labels])
    keep_idx = []
    for cls in labels.unique():
        cls_mask = (labels == cls).nonzero(as_tuple=True)[0]
        if len(cls_mask) == 0:
            continue
        nms_keep = nms(bboxes[cls_mask], scores[cls_mask], args.nms_iou)
        keep_idx.append(cls_mask[nms_keep])
    keep = torch.cat(keep_idx) if keep_idx else torch.zeros(0, dtype=torch.long)
    results.append(dict(
        img_id=id_by_name[os.path.basename(path)],
        img_path=path,
        ori_shape=(H, W),
        pred_instances=dict(
            bboxes=bboxes[keep].cpu(),
            scores=scores[keep].cpu(),
            labels=labels[keep].cpu(),
        ),
    ))

mmengine.dump(results, args.out)
print(f'TTA done: {len(results)} images, out={args.out}')

In [ ]:
import time; PIPELINE_T0 = time.time()

import os, sys, subprocess, traceback

LOG_PATH = '/kaggle/working/progress.log'
ERR_PATH = '/kaggle/working/error.log'

def log_stage(name, info=''):
    try:
        import psutil
        ram = psutil.virtual_memory()
        ram_str = f'ram={ram.used/1024**3:.1f}/{ram.total/1024**3:.1f}GB'
    except Exception:
        ram_str = 'ram=?'
    try:
        import torch
        if torch.cuda.is_available():
            parts = []
            for i in range(torch.cuda.device_count()):
                alloc = torch.cuda.memory_allocated(i) / 1024**3
                reserv = torch.cuda.memory_reserved(i) / 1024**3
                parts.append(f'gpu{i}={alloc:.1f}/{reserv:.1f}GB')
            gpu_str = ' '.join(parts)
        else:
            gpu_str = 'no cuda'
    except Exception:
        gpu_str = 'gpu=?'
    try:
        s = os.statvfs('/kaggle/working')
        disk_str = f'disk_free={s.f_bavail * s.f_frsize / 1024**3:.1f}GB'
    except Exception:
        disk_str = 'disk=?'
    line = f'[{time.strftime("%H:%M:%S")} +{(time.time()-PIPELINE_T0)/60:.1f}m] {name} | {ram_str} | {gpu_str} | {disk_str}'
    if info:
        line += f' | {info}'
    with open(LOG_PATH, 'a') as f:
        f.write(line + '\n')
    print(line, flush=True)

def _excepthook(exc_type, exc_value, exc_tb):
    with open(ERR_PATH, 'a') as f:
        f.write(f'\n=== UNCAUGHT EXCEPTION at {time.strftime("%H:%M:%S")} ===\n')
        traceback.print_exception(exc_type, exc_value, exc_tb, file=f)
    sys.__excepthook__(exc_type, exc_value, exc_tb)
sys.excepthook = _excepthook

def _run_2gpu(args0, args1, role='mmdet'):
    log_stage(f'START {role}', f'args0={args0[-2:]} args1={args1[-2:]}')
    err0_path = f'/kaggle/working/stderr_{role}_g0.log'
    err1_path = f'/kaggle/working/stderr_{role}_g1.log'
    env0 = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0'}
    env1 = {**os.environ, 'CUDA_VISIBLE_DEVICES': '1'}
    with open(err0_path, 'wb') as e0, open(err1_path, 'wb') as e1:
        p0 = subprocess.Popen(args0, env=env0, stderr=e0)
        p1 = subprocess.Popen(args1, env=env1, stderr=e1)
        p0.wait(); p1.wait()
    log_stage(f'END {role}', f'rc0={p0.returncode} rc1={p1.returncode}')
    if p0.returncode or p1.returncode:
        with open(ERR_PATH, 'a') as out:
            for path in [err0_path, err1_path]:
                if os.path.exists(path):
                    out.write(f'\n=== {path} ===\n')
                    with open(path) as f:
                        out.write(f.read())
        raise RuntimeError(f'{role} subprocess failed: rc0={p0.returncode} rc1={p1.returncode}')

def test_pair(cfg, ck0, ck1, out0, out1):
    role = os.path.basename(out0).replace('.pkl', '') + '+' + os.path.basename(out1).replace('.pkl', '')
    _run_2gpu(
        ['python', 'tta_test.py', cfg, ck0, '--out', out0],
        ['python', 'tta_test.py', cfg, ck1, '--out', out1],
        role=role,
    )

for p in [LOG_PATH, ERR_PATH]:
    if os.path.exists(p):
        os.remove(p)
log_stage('pipeline_start')

## Run the 8 legacy mmdet streams (1st-place ckpts, m0i/m1i dropped)

In [ ]:
test_pair(
    '/kaggle/working/inputs/hubmap-2023-configs-patched/r0i.py',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/r0i.pth',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/r1i.pth',
    '/kaggle/working/r0i.pkl',
    '/kaggle/working/r1i.pkl',
)

In [ ]:
test_pair(
    '/kaggle/working/inputs/hubmap-2023-configs-patched/s0i.py',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/s0i.pth',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/s1i.pth',
    '/kaggle/working/s0i.pkl',
    '/kaggle/working/s1i.pkl',
)

In [ ]:
test_pair(
    '/kaggle/working/inputs/hubmap-2023-configs-patched/sb0i.py',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/sb0i.pth',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/sb1i.pth',
    '/kaggle/working/sb0i.pkl',
    '/kaggle/working/sb1i.pkl',
)

In [ ]:
test_pair(
    '/kaggle/working/inputs/hubmap-2023-configs-patched/y0i.py',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/y0i.pth',
    '/kaggle/working/inputs/hubmap-2023-checkpoints/y1i.pth',
    '/kaggle/working/y0i.pkl',
    '/kaggle/working/y1i.pkl',
)

## Run the 2 new Cascade Mask R-CNN + Swin-L streams (replaces m0i/m1i)

In [ ]:
# The new cascade dataset must contain:
#   m0ci.py        (inference config — Kaggle paths)
#   m0c_fold0.pth  (EMA-dumped weights from tools/dump_ckpt.py, fold0 run)
#   m0c_fold1.pth  (likewise, fold1 run)
test_pair(
    '/kaggle/working/inputs/hubmap-cascade-ckpts/m0ci.py',
    '/kaggle/working/inputs/hubmap-cascade-ckpts/m0c_fold0.pth',
    '/kaggle/working/inputs/hubmap-cascade-ckpts/m0c_fold1.pth',
    '/kaggle/working/cascade0.pkl',
    '/kaggle/working/cascade1.pkl',
)

## WBF ensemble

In [ ]:
log_stage('START wbf')
import torch
import mmengine
from ensemble_boxes import weighted_boxes_fusion

# ABLATION_MODE:
#   'cascade_swap'  = 8 legacy mmdet (no m0i/m1i) + 2 cascade  [primary, what we're testing]
#   'mmdet_only_legacy' = full 10 legacy mmdet stream (matches 0.589 baseline) — A/B reference
#   'cascade_only'  = just the 2 cascade streams (sanity / single-model floor)
ABLATION_MODE = 'cascade_swap'

LEGACY_8_NAMES   = ['r0i','r1i','s0i','s1i','y0i','y1i','sb0i','sb1i']
LEGACY_8_WEIGHTS = [2,2,2,2,1,1,2,2]

LEGACY_10_NAMES   = ['r0i','r1i','s0i','s1i','m0i','m1i','y0i','y1i','sb0i','sb1i']
LEGACY_10_WEIGHTS = [2,2,2,2,1,1,1,1,2,2]

CASCADE_NAMES   = ['cascade0','cascade1']
CASCADE_WEIGHTS = [2,2]

if ABLATION_MODE == 'cascade_swap':
    names = LEGACY_8_NAMES + CASCADE_NAMES
    weights = LEGACY_8_WEIGHTS + CASCADE_WEIGHTS
elif ABLATION_MODE == 'mmdet_only_legacy':
    names, weights = LEGACY_10_NAMES, LEGACY_10_WEIGHTS
elif ABLATION_MODE == 'cascade_only':
    names, weights = CASCADE_NAMES, CASCADE_WEIGHTS
else:
    raise ValueError(f'unknown ABLATION_MODE: {ABLATION_MODE}')

print(f'WBF mode={ABLATION_MODE}, streams={len(names)}: {names}')

results = [mmengine.load(f'/kaggle/working/{n}.pkl') for n in names]

SCALER  = 10000
IOU_THR = 0.7  # same as 1st place

for rs in zip(*results):
    boxes_list  = [(r['pred_instances']['bboxes'] / SCALER).tolist() for r in rs]
    scores_list = [r['pred_instances']['scores'].tolist()           for r in rs]
    labels_list = [r['pred_instances']['labels'].tolist()           for r in rs]
    boxes, scores, labels = weighted_boxes_fusion(
        boxes_list, scores_list, labels_list,
        weights=weights, iou_thr=IOU_THR, conf_type='avg',
    )
    rs[0]['pred_instances'] = dict(
        bboxes=torch.from_numpy(boxes).float() * SCALER,
        scores=torch.from_numpy(scores).float(),
        labels=torch.from_numpy(labels).long(),
    )

mmengine.dump(results[0], 'ensemble.pkl')
log_stage('END wbf')

## Mask repredict at 1440x1440 using the cascade fold-0 model

Resamples WBF boxes to high-resolution canvas and runs `roi_head.predict_mask` to get refined masks. We use the cascade fold0 ckpt because it's the strongest mask model in this ensemble (FCNMaskHead in cascade stage-3).

In [ ]:
%%writefile predict_mask.py
import mmcv
import mmengine
import torch
from mmengine.runner import load_checkpoint
from mmengine.structures.instance_data import InstanceData
from mmdet.registry import MODELS
from mmdet.structures import DetDataSample
from mmdet.structures.mask import encode_mask_results
from mmdet.utils import register_all_modules

register_all_modules()
cfg = mmengine.Config.fromfile('/kaggle/working/inputs/hubmap-cascade-ckpts/m0ci.py')
model = MODELS.build(cfg.model)
load_checkpoint(model, '/kaggle/working/inputs/hubmap-cascade-ckpts/m0c_fold0.pth')
model.eval()
model.cuda()


@torch.no_grad()
def predict_mask(result, input_size=(1440, 1440)):
    img = mmcv.imread(result['img_path'])
    img = mmcv.imresize(img, input_size)
    batch_data = dict(
        inputs=torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).cuda(),
        data_samples=[
            DetDataSample(metainfo=dict(img_id=result['img_id'],
                                        ori_shape=(512, 512),
                                        img_shape=(1440, 1440),
                                        img_path=result['img_path'],
                                        scale_factor=(1440 / 512, 1440 / 512)))
        ])
    batch_data = model.data_preprocessor(batch_data, False)
    batch_data_inputs = batch_data['inputs']
    batch_data_samples = batch_data['data_samples']
    batch_img_metas = [ds.metainfo for ds in batch_data_samples]
    img_feats = model.extract_feat(batch_data_inputs)

    img_result = InstanceData()
    for k, v in result['pred_instances'].items():
        img_result[k] = v.cuda()
    img_result.bboxes *= 1440 / 512
    results_list = model.roi_head.predict_mask(
        img_feats, batch_img_metas, [img_result], rescale=True)
    out = results_list[0].cpu()
    return dict(
        img_id=result['img_id'],
        ori_shape=(512, 512),
        img_shape=(1440, 1440),
        img_path=result['img_path'],
        scale_factor=(1440 / 512, 1440 / 512),
        pred_instances=dict(
            bboxes=out['bboxes'],
            labels=out['labels'],
            scores=out['scores'],
            masks=encode_mask_results(out['masks']),
        ),
    )


results = mmengine.load('/kaggle/working/ensemble.pkl')
outputs = [predict_mask(r) for r in results]
mmengine.dump(outputs, '/kaggle/working/ensemble_results.pkl')

In [ ]:
log_stage('START predict_mask')
_proc = subprocess.run(
    ['python', 'predict_mask.py'],
    stderr=subprocess.PIPE,
)
with open('/kaggle/working/stderr_predict_mask.log', 'wb') as f:
    f.write(_proc.stderr or b'')
if _proc.returncode != 0:
    with open(ERR_PATH, 'a') as out:
        out.write('\n=== predict_mask.py stderr ===\n')
        out.write((_proc.stderr or b'').decode(errors='replace'))
    raise RuntimeError(f'predict_mask.py exit {_proc.returncode}')
log_stage('END predict_mask')

## RLE encode + submission

In [ ]:
import base64
import numpy as np
from pycocotools import _mask as coco_mask
import typing as t
import zlib


def encode_binary_mask(mask: np.ndarray) -> t.Text:
    if mask.dtype != bool:
        raise ValueError(f'encode_binary_mask expects bool mask, got {mask.dtype}')
    mask = np.squeeze(mask)
    if len(mask.shape) != 2:
        raise ValueError(f'encode_binary_mask expects 2d mask, got {mask.shape}')
    mask_to_encode = mask.reshape(mask.shape[0], mask.shape[1], 1).astype(np.uint8)
    mask_to_encode = np.asfortranarray(mask_to_encode)
    encoded_mask = coco_mask.encode(mask_to_encode)[0]['counts']
    binary_str = zlib.compress(encoded_mask, zlib.Z_BEST_COMPRESSION)
    return base64.b64encode(binary_str)

In [ ]:
log_stage('START encode_submission')
import os
import mmengine
import pandas as pd
import pycocotools.mask as mask_utils

results = mmengine.load('/kaggle/working/ensemble_results.pkl')
ids = []
HEIGHT = 512
WIDTH = 512
prediction_strings = []
for result in results:
    img_path = result['img_path']
    filename = os.path.basename(img_path)
    ids.append(filename[:-4])
    pred_instances = result['pred_instances']
    scores = pred_instances['scores'].tolist()
    labels = pred_instances['labels'].tolist()
    masks = pred_instances['masks']
    instance_strings = []
    for label, score, mask in zip(labels, scores, masks):
        if label != 0:
            continue
        mask = mask_utils.decode(mask).astype(bool)
        mask_string = encode_binary_mask(mask).decode('utf-8')
        instance_strings.append(f'{label} {score} {mask_string}')
    prediction_strings.append(' '.join(instance_strings))
log_stage('END encode_submission')

In [ ]:
sub = pd.DataFrame(dict(
    id=ids,
    height=[HEIGHT] * len(ids),
    width=[WIDTH] * len(ids),
    prediction_string=prediction_strings,
))
sub.to_csv('submission.csv', index=False)
import time as _t
print(f'Total pipeline elapsed: {(_t.time() - PIPELINE_T0) / 3600:.2f} hours')
log_stage('pipeline_done')
sub.head()